# Notebook 01 — SQLi Detector: Training & Benchmarking

**Fixes vs original:**
- **Leakage-proof**: split raw text first → `fit(train)` only → `transform(test)`
- **Threshold search corrected**: scan PR curve from HIGH→LOW so T_high > T_low always
  - T_high = highest threshold where precision ≥ 0.99 (ATTACK tier)
  - T_low  = highest threshold where recall  ≥ 0.95  (SUSPICIOUS tier)
- SGDClassifier added (production-friendly alternative to LR)
- CalibratedClassifierCV wraps LinearSVC for probability outputs
- 5-fold stratified CV with vectorizer refit per fold (no leakage)
- PR-AUC alongside F1; inference latency per model
- KNN excluded (9,792 ms — incompatible with real-time)


## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import joblib, os, time, json, warnings
warnings.filterwarnings('ignore')

from collections import Counter
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    average_precision_score, precision_recall_curve
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

for d in ['results/models', 'results/figures', 'results/metrics']:
    os.makedirs(d, exist_ok=True)

print('Setup complete.')


Setup complete.


## 2. Load Dataset

In [2]:
df = pd.read_csv('../datasets/Cleaned_SQL_Dataset.csv')
print(f'Shape   : {df.shape}')
print(f'Columns : {df.columns.tolist()}')
print()
vc = df['Label'].value_counts().rename({0: 'legitimate (0)', 1: 'malicious (1)'})
print('Label distribution:')
print(vc.to_string())
print()
print('Sample malicious:')
for q in df[df['Label'] == 1]['Query'].head(3):
    print(f'  {str(q)[:100]}')
print()
print('Sample legitimate:')
for q in df[df['Label'] == 0]['Query'].head(3):
    print(f'  {str(q)[:100]}')


Shape   : (30924, 2)
Columns : ['Query', 'Label']

Label distribution:
Label
legitimate (0)    19537
malicious (1)     11387

Sample malicious:
  " or pg_sleep  (  __TIME__  )  --
  create user name identified by pass123 temporary tablespace temp default tablespace users;
   AND 1  =  utl_inaddr.get_host_address   (    (   SELECT DISTINCT  (  table_name  )   FROM   (  SELE

Sample legitimate:
  99745017c
  ejerci78
  47209


## 3. Feature Helpers

In [3]:
SYMBOLS = ["'",'"',";","--","#","/*","*/","*","+","|","(",")",">","<","\\","/","="]

def extract_symbol_frequencies(text):
    c = Counter()
    for sym in SYMBOLS:
        c[sym] = str(text).count(sym)
    return c

def build_symbol_matrix(queries):
    rows = [extract_symbol_frequencies(q) for q in queries]
    return csr_matrix(pd.DataFrame(rows).fillna(0).values)

test_q = "1' UNION SELECT null, username, password FROM users--"
print(f'Sanity check: {test_q}')
print(f'Symbols: {dict(extract_symbol_frequencies(test_q))}')


Sanity check: 1' UNION SELECT null, username, password FROM users--
Symbols: {"'": 1, '"': 0, ';': 0, '--': 1, '#': 0, '/*': 0, '*/': 0, '*': 0, '+': 0, '|': 0, '(': 0, ')': 0, '>': 0, '<': 0, '\\': 0, '/': 0, '=': 0}


## 4. Split First, Then Vectorize

**Critical fix:** raw text strings are split 80/20 *before* any vectorization.
The CountVectorizer is then fitted on the training split only, preventing
test-set vocabulary from leaking into the feature space.


In [4]:
queries = df['Query'].fillna('').tolist()
labels  = df['Label'].values

# STEP 1: split raw text — no features yet
q_train, q_test, y_train, y_test = train_test_split(
    queries, labels, test_size=0.2, random_state=42, stratify=labels)

# STEP 2: fit ONLY on train
vectorizer = CountVectorizer(analyzer='char', ngram_range=(1, 3))
X_train = hstack([vectorizer.fit_transform(q_train), build_symbol_matrix(q_train)])
X_test  = hstack([vectorizer.transform(q_test),      build_symbol_matrix(q_test)])
y_train = np.array(y_train)
y_test  = np.array(y_test)

print(f'Train  : {len(q_train):,}  pos={int((y_train==1).sum()):,}  neg={int((y_train==0).sum()):,}')
print(f'Test   : {len(q_test):,}   pos={int((y_test==1).sum()):,}   neg={int((y_test==0).sum()):,}')
print(f'Vocab  : {len(vectorizer.vocabulary_):,} char n-grams (train-only) + {len(SYMBOLS)} symbols')
print(f'Shape  : {X_train.shape}')

joblib.dump(vectorizer, 'results/models/vectorizer_with_symbols.pkl')
print('Saved: results/models/vectorizer_with_symbols.pkl')


Train  : 24,739  pos=9,110  neg=15,629
Test   : 6,185   pos=2,277   neg=3,908
Vocab  : 29,535 char n-grams (train-only) + 17 symbols
Shape  : (24739, 29552)
Saved: results/models/vectorizer_with_symbols.pkl


## 5. Train & Evaluate Models

In [5]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SGD (log loss)':      SGDClassifier(loss='log_loss', max_iter=1000, random_state=42),
    'LinearSVC':           CalibratedClassifierCV(LinearSVC(max_iter=2000)),
    'Decision Tree':       DecisionTreeClassifier(),
    'Naive Bayes':         MultinomialNB(),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}
# KNN excluded: 9,792 ms inference — incompatible with real-time requirement

results, cms, pr_data, trained = [], {}, {}, {}

for name, model in models.items():
    print(f'Training {name}...', end=' ', flush=True)
    model.fit(X_train, y_train)

    t0     = time.perf_counter()
    y_pred = model.predict(X_test)
    lat_ms = (time.perf_counter() - t0) * 1000
    y_prob = model.predict_proba(X_test)[:, 1]

    f1    = f1_score(y_test, y_pred, zero_division=0)
    prauc = average_precision_score(y_test, y_prob)
    p_c, r_c, thr = precision_recall_curve(y_test, y_prob)

    results.append({
        'Model':      name,
        'Accuracy':   round(accuracy_score(y_test, y_pred), 4),
        'Precision':  round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall':     round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1-score':   round(f1, 4),
        'PR-AUC':     round(prauc, 4),
        'Latency_ms': round(lat_ms, 1),
    })
    cms[name]     = confusion_matrix(y_test, y_pred)
    pr_data[name] = (p_c, r_c, thr, prauc)
    trained[name] = (model, y_prob)

    fname = name.replace(' ', '_').replace('(', '').replace(')', '').lower()
    joblib.dump(model, f'results/models/{fname}_model_with_symbols.pkl')
    print(f'F1={f1:.4f}  PR-AUC={prauc:.4f}  Lat={lat_ms:.1f}ms  saved')

results_df = pd.DataFrame(results)
results_df.to_csv('results/metrics/01_model_results.csv', index=False)
print()
print('=== TEST SET RESULTS ===')
print(results_df.to_string(index=False))


Training Logistic Regression... F1=0.9947  PR-AUC=0.9989  Lat=2.2ms  saved
Training SGD (log loss)... F1=0.9914  PR-AUC=0.9985  Lat=2.3ms  saved
Training LinearSVC... F1=0.9938  PR-AUC=0.9990  Lat=15.6ms  saved
Training Decision Tree... F1=0.9919  PR-AUC=0.9879  Lat=11.2ms  saved
Training Naive Bayes... F1=0.9164  PR-AUC=0.9904  Lat=5.0ms  saved
Training Random Forest... F1=0.9956  PR-AUC=0.9991  Lat=97.6ms  saved

=== TEST SET RESULTS ===
              Model  Accuracy  Precision  Recall  F1-score  PR-AUC  Latency_ms
Logistic Regression    0.9961     1.0000  0.9895    0.9947  0.9989         2.2
     SGD (log loss)    0.9937     0.9973  0.9855    0.9914  0.9985         2.3
          LinearSVC    0.9955     0.9991  0.9886    0.9938  0.9990        15.6
      Decision Tree    0.9940     0.9934  0.9903    0.9919  0.9879        11.2
        Naive Bayes    0.9342     0.8607  0.9798    0.9164  0.9904         5.0
      Random Forest    0.9968     1.0000  0.9912    0.9956  0.9991        97.6


## 6. Threshold Tuning — Corrected Direction

Scanning the PR curve from **high threshold downward** ensures:
- `T_high` = strictest threshold that still achieves precision ≥ 0.99
- `T_low`  = strictest threshold that still achieves recall ≥ 0.95
- T_high > T_low by construction → SUSPICIOUS band is non-empty


In [6]:
print('=== THRESHOLD TUNING (Random Forest) ===')
print()
rf_probs       = trained['Random Forest'][1]
p_c, r_c, thr, _ = pr_data['Random Forest']

# Scan from HIGH threshold downward — guarantees T_high > T_low
pairs = sorted(zip(thr, p_c[:-1], r_c[:-1]), key=lambda x: -x[0])

t_high = p_high = r_high = None
for t, p, r in pairs:
    if p >= 0.995:
        t_high, p_high, r_high = round(float(t),3), round(float(p),4), round(float(r),4)
        break

t_low = p_low = r_low = None
for t, p, r in pairs:
    if r >= 0.95:
        t_low, p_low, r_low = round(float(t),3), round(float(p),4), round(float(r),4)
        break

assert t_high is not None, "No threshold found with precision >= 0.99"
assert t_low  is not None, "No threshold found with recall >= 0.95"
assert t_high > t_low, f"T_high ({t_high}) must be > T_low ({t_low})"

print(f'T_high (highest t where precision>=0.99) : {t_high}  prec={p_high}  rec={r_high}')
print(f'T_low  (highest t where recall>=0.95)    : {t_low}   prec={p_low}   rec={r_low}')
print(f'T_high > T_low : {t_high} > {t_low}  ✅')
print()

BENIGN_TEST = int((y_test == 0).sum())
for lbl, tv in [('T_high', t_high), ('T_low', t_low)]:
    yp = (rf_probs >= tv).astype(int)
    tp = int(((yp==1)&(y_test==1)).sum())
    fp = int(((yp==1)&(y_test==0)).sum())
    fn = int(((yp==0)&(y_test==1)).sum())
    print(f'At {lbl}={tv}: TP={tp}  FP={fp}  FN={fn}  FP/10k(test)={fp/BENIGN_TEST*10000:.1f}')

th_dict = {
    't_high': t_high, 'p_high': p_high, 'r_high': r_high,
    't_low':  t_low,  'p_low':  p_low,  'r_low':  r_low,
}
with open('results/models/01_thresholds.json', 'w') as f:
    json.dump(th_dict, f, indent=2)
print()
print('Saved: results/models/01_thresholds.json')


=== THRESHOLD TUNING (Random Forest) ===

T_high (highest t where precision>=0.99) : 1.0  prec=1.0  rec=0.6961
T_low  (highest t where recall>=0.95)    : 0.96   prec=1.0   rec=0.9521
T_high > T_low : 1.0 > 0.96  ✅

At T_high=1.0: TP=1585  FP=0  FN=692  FP/10k(test)=0.0
At T_low=0.96: TP=2168  FP=0  FN=109  FP/10k(test)=0.0

Saved: results/models/01_thresholds.json


## 7. 5-Fold Stratified Cross-Validation

In [7]:
print('Running 5-fold CV (vectorizer refit per fold — no leakage)...')
print()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
qa, la = np.array(queries), np.array(labels)

cv_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SGD (log loss)':      SGDClassifier(loss='log_loss', max_iter=1000, random_state=42),
    'LinearSVC':           CalibratedClassifierCV(LinearSVC(max_iter=2000)),
    'Decision Tree':       DecisionTreeClassifier(),
    'Naive Bayes':         MultinomialNB(),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

cv_rows = []
for name, model in cv_models.items():
    fold_f1, fold_pr = [], []
    for tr_i, va_i in skf.split(qa, la):
        v   = CountVectorizer(analyzer='char', ngram_range=(1, 3))
        Xtr = hstack([v.fit_transform(qa[tr_i].tolist()), build_symbol_matrix(qa[tr_i].tolist())])
        Xva = hstack([v.transform(qa[va_i].tolist()),     build_symbol_matrix(qa[va_i].tolist())])
        model.fit(Xtr, la[tr_i])
        yp   = model.predict(Xva)
        yprb = model.predict_proba(Xva)[:, 1]
        fold_f1.append(f1_score(la[va_i], yp, zero_division=0))
        fold_pr.append(average_precision_score(la[va_i], yprb))

    cv_rows.append({
        'Model':         name,
        'CV_F1_mean':    round(np.mean(fold_f1), 4),
        'CV_F1_std':     round(np.std(fold_f1),  4),
        'CV_PRAUC_mean': round(np.mean(fold_pr),  4),
        'CV_PRAUC_std':  round(np.std(fold_pr),   4),
    })
    print(f'  {name:25s}: F1={np.mean(fold_f1):.4f}±{np.std(fold_f1):.4f}'
          f'  PR-AUC={np.mean(fold_pr):.4f}±{np.std(fold_pr):.4f}')

cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv('results/metrics/01_cv_results.csv', index=False)
print()
print(cv_df.to_string(index=False))


Running 5-fold CV (vectorizer refit per fold — no leakage)...

  Logistic Regression      : F1=0.9955±0.0006  PR-AUC=0.9992±0.0006
  SGD (log loss)           : F1=0.9940±0.0009  PR-AUC=0.9991±0.0002
  LinearSVC                : F1=0.9941±0.0002  PR-AUC=0.9988±0.0008
  Decision Tree            : F1=0.9903±0.0010  PR-AUC=0.9839±0.0022
  Naive Bayes              : F1=0.9175±0.0043  PR-AUC=0.9901±0.0010
  Random Forest            : F1=0.9965±0.0005  PR-AUC=0.9995±0.0002

              Model  CV_F1_mean  CV_F1_std  CV_PRAUC_mean  CV_PRAUC_std
Logistic Regression      0.9955     0.0006         0.9992        0.0006
     SGD (log loss)      0.9940     0.0009         0.9991        0.0002
          LinearSVC      0.9941     0.0002         0.9988        0.0008
      Decision Tree      0.9903     0.0010         0.9839        0.0022
        Naive Bayes      0.9175     0.0043         0.9901        0.0010
      Random Forest      0.9965     0.0005         0.9995        0.0002


## 8. Visualisations

In [8]:
# --- confusion matrices ---
fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4))
for ax, (name, cm) in zip(axes, cms.items()):
    ConfusionMatrixDisplay(cm, display_labels=['legit', 'attack']).plot(
        cmap=plt.cm.Blues, ax=ax, colorbar=False)
    ax.set_title(name, fontsize=8)
plt.suptitle('Confusion Matrices — NB01 Test Set', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('results/figures/01_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.close()

# --- PR curves ---
fig, ax = plt.subplots(figsize=(9, 6))
for name, (p, r, t, auc) in pr_data.items():
    ax.plot(r, p, label=f'{name} (AUC={auc:.4f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves — NB01')
ax.legend(loc='lower left', fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/figures/01_pr_curves.png', dpi=150)
plt.close()

# --- F1 + latency ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
b1 = ax1.bar(results_df['Model'], results_df['F1-score'], color='steelblue')
ax1.set_ylim(0, 1.05); ax1.set_ylabel('F1-score'); ax1.set_title('F1-Score')
ax1.grid(axis='y', alpha=0.4); ax1.tick_params(axis='x', rotation=20)
for b, v in zip(b1, results_df['F1-score']):
    ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
             f'{v:.4f}', ha='center', fontsize=7)
b2 = ax2.bar(results_df['Model'], results_df['Latency_ms'], color='coral')
ax2.set_ylabel('Latency (ms)'); ax2.set_title('Inference Latency (test set)')
ax2.grid(axis='y', alpha=0.4); ax2.tick_params(axis='x', rotation=20)
for b, v in zip(b2, results_df['Latency_ms']):
    ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 0.2,
             f'{v:.1f}ms', ha='center', fontsize=7)
plt.tight_layout()
plt.savefig('results/figures/01_f1_and_latency.png', dpi=150)
plt.close()

print('Saved: 01_confusion_matrices.png / 01_pr_curves.png / 01_f1_and_latency.png')


Saved: 01_confusion_matrices.png / 01_pr_curves.png / 01_f1_and_latency.png


## 9. Summary

In [9]:
th = json.load(open('results/models/01_thresholds.json'))
print('=' * 65)
print('NOTEBOOK 01 — COMPLETE')
print('=' * 65)
print(f'Dataset : {df.shape[0]:,} rows'
      f' | {int((labels==1).sum()):,} malicious'
      f' | {int((labels==0).sum()):,} legitimate')
print(f'Split   : {len(q_train):,} train / {len(q_test):,} test (80/20 stratified)')
print(f'Vocab   : {len(vectorizer.vocabulary_):,} char n-grams (train-only) + {len(SYMBOLS)} symbols')
print()
print('TEST SET RESULTS:')
print(results_df[['Model', 'F1-score', 'PR-AUC', 'Latency_ms']].to_string(index=False))
print()
print('5-FOLD CV:')
print(cv_df[['Model', 'CV_F1_mean', 'CV_F1_std', 'CV_PRAUC_mean']].to_string(index=False))
print()
print(f'THRESHOLDS (RF):')
print(f'  T_high = {th["t_high"]}  precision={th["p_high"]}  recall={th["r_high"]}')
print(f'  T_low  = {th["t_low"]}   precision={th["p_low"]}   recall={th["r_low"]}')
print(f'  T_high > T_low : {th["t_high"]} > {th["t_low"]}  ✅')
print()
print('NEXT: Notebook 02 — Baseline evaluation on real access logs')


NOTEBOOK 01 — COMPLETE
Dataset : 30,924 rows | 11,387 malicious | 19,537 legitimate
Split   : 24,739 train / 6,185 test (80/20 stratified)
Vocab   : 29,535 char n-grams (train-only) + 17 symbols

TEST SET RESULTS:
              Model  F1-score  PR-AUC  Latency_ms
Logistic Regression    0.9947  0.9989         2.2
     SGD (log loss)    0.9914  0.9985         2.3
          LinearSVC    0.9938  0.9990        15.6
      Decision Tree    0.9919  0.9879        11.2
        Naive Bayes    0.9164  0.9904         5.0
      Random Forest    0.9956  0.9991        97.6

5-FOLD CV:
              Model  CV_F1_mean  CV_F1_std  CV_PRAUC_mean
Logistic Regression      0.9955     0.0006         0.9992
     SGD (log loss)      0.9940     0.0009         0.9991
          LinearSVC      0.9941     0.0002         0.9988
      Decision Tree      0.9903     0.0010         0.9839
        Naive Bayes      0.9175     0.0043         0.9901
      Random Forest      0.9965     0.0005         0.9995

THRESHOLDS (RF):
